# OneDrive to Google Drive Transfer (Simplified)
# OneDrive到Google Drive传输（简化版）

**This version uses manual token input to avoid browser issues**

**本版本使用手动输入token以避免浏览器问题**

---

## Step 0: Prerequisites (Do this on YOUR COMPUTER first)
## 步骤0：前置条件（先在你的电脑上完成）

### On your LOCAL Windows computer:

1. **Download rclone:**
   - Go to: https://rclone.org/downloads/
   - Download Windows version
   - Extract to a folder (e.g., `C:\rclone`)

2. **Get OneDrive token:**
   ```
   cd C:\rclone
   .\rclone.exe authorize "onedrive"
   ```
   - Browser will open
   - Login with your UNC account
   - Copy the entire token (JSON string)

3. **Keep the token ready to paste below**

---

## Step 1: Install Rclone
## 步骤1：安装Rclone

In [ ]:
# Install rclone
!curl https://rclone.org/install.sh | sudo bash

# Verify installation
!rclone version

print("\n✅ Rclone installed successfully!")

## Step 2: Mount Google Drive
## 步骤2：挂载Google Drive

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted!")

## Step 3: Configure OneDrive (Manual Token)
## 步骤3：配置OneDrive（手动Token）

**Paste the token you got from your local computer**

In [ ]:
import subprocess

print("🔐 OneDrive Configuration")
print("=" * 70)
print("\nPaste your OneDrive authorization token below.")
print("(The JSON string you got from: rclone authorize \"onedrive\")")
print("\nIt should look like:")
print('{"access_token":"xxx","token_type":"Bearer",...}')
print("=" * 70)
print()

# Get token from user
onedrive_token = input("Paste OneDrive token: ").strip()

if not onedrive_token:
    raise ValueError("❌ No token provided!")

# Create rclone config file
config_content = f"""[unc_onedrive]
type = onedrive
token = {onedrive_token}
drive_type = business
"""

with open('/content/rclone.conf', 'w') as f:
    f.write(config_content)

print("\n✅ OneDrive configured!")
print("\n📋 Testing connection...")

# Test connection
result = subprocess.run(
    ['rclone', 'lsd', 'unc_onedrive:', '--config', '/content/rclone.conf'],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("\n✅ Connection successful!")
    print("\n📁 Folders in OneDrive:")
    print(result.stdout)
else:
    print("\n❌ Connection failed!")
    print(result.stderr)
    raise Exception("OneDrive connection failed. Please check your token.")

## Step 4: Verify Source Files
## 步骤4：验证源文件

In [ ]:
print("📂 Checking Egolife_videos folder in OneDrive...\n")

# List files (first 10)
!rclone ls unc_onedrive:Egolife_videos --config /content/rclone.conf --max-depth 1 | head -n 10

# Count total files
result = !rclone ls unc_onedrive:Egolife_videos --config /content/rclone.conf | wc -l
print(f"\n✅ Total files found: {result[0]}")
print("   Expected: 240 files")

# Check total size
print("\n📊 Total size:")
!rclone size unc_onedrive:Egolife_videos --config /content/rclone.conf

## Step 5: Create Destination Folder
## 步骤5：创建目标文件夹

In [ ]:
!mkdir -p "/content/drive/MyDrive/Egolife_videos"

print("✅ Destination folder created: /content/drive/MyDrive/Egolife_videos")

## Step 6: START TRANSFER ⚡
## 步骤6：开始传输 ⚡

**This will take 2-4 hours for 481GB**

**Progress updates every 30 seconds**

In [ ]:
from datetime import datetime

print("🚀 Starting transfer...")
print(f"⏰ Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n" + "="*70)
print("TRANSFERRING 481GB (240 videos)")
print("Estimated time: 2-4 hours")
print("Progress updates every 30 seconds")
print("="*70 + "\n")

# Start transfer
!rclone copy \
  unc_onedrive:Egolife_videos \
  "/content/drive/MyDrive/Egolife_videos" \
  --config /content/rclone.conf \
  --progress \
  --transfers 8 \
  --checkers 8 \
  --buffer-size 128M \
  --drive-chunk-size 64M \
  --stats 30s \
  --stats-one-line \
  --log-file /content/transfer.log \
  -v

print("\n" + "="*70)
print("✅ TRANSFER COMPLETE!")
print(f"⏰ End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

## Step 7: Verify Transfer
## 步骤7：验证传输

In [ ]:
print("📊 Verifying transfer...\n")

# Count files
file_count = !ls "/content/drive/MyDrive/Egolife_videos" | wc -l
print(f"✅ Files transferred: {file_count[0]}")
print("   Expected: 240 files\n")

# Check total size
!du -sh "/content/drive/MyDrive/Egolife_videos"

# List first 10 files
print("\n📁 First 10 files:")
!ls "/content/drive/MyDrive/Egolife_videos" | head -n 10

if int(file_count[0]) == 240:
    print("\n" + "="*70)
    print("✅ ✅ ✅ TRANSFER SUCCESSFUL! ✅ ✅ ✅")
    print("All 240 videos transferred to Google Drive!")
    print("="*70)
else:
    print(f"\n⚠️  Warning: Expected 240 files, found {file_count[0]}")
    print("Check the transfer log for errors.")

## ✅ Done!

**Your videos are now in Google Drive!**

**Next:** Contact me to generate public URLs for all videos.